# Neural Surrogate Policy — Frictionless Validation Notebook

Validates the β-conditional neural surrogate `φ_NN(k, z, β)` on the
**frictionless basic investment model** (Bayesian.md §3, special case of
model #2 in §1.3 with `φ_quad = φ_prop = 0`). Strebulaev–Whited (2012)
§3.1 gives a closed-form `k'(z; β)` we use as ground truth.

**β coordinates.** β is 5-dim `(α, ρ, σ_ε, φ_quad, φ_prop)`. This
notebook freezes `φ_quad = φ_prop = 0` via `BetaSampler(freeze_dims=(3, 4))`,
so effective β here is 3-dim.

**Training distribution ≠ inference prior.** Training draws β
**uniformly** over each coordinate's box (`mode="uniform"`). The
Bayesian.md §2.5 priors (Beta(2,2), HalfNormal) are 3–4× sparser at the
validation-sweep tails than at the bulk; uniform training gives the
surrogate even accuracy across the support. At MCMC time (Phase B) the
literal priors are still used — the surrogate is asked for
`φ_NN(k, z, β)` at whatever β the chain visits, regardless of how the
surrogate was trained.

**Reproducibility.** β is *not* baked into the dataset — fresh draws per
minibatch via a per-step seed derived from `master_seed`. Determinism is
pinned by `test_per_step_beta_seed_is_deterministic` in
`src/v2/tests/test_er_param.py`.

**Gates.**
1. Held-out MAE of `k'` vs analytical, evaluated on a fixed (s, β) set.
2. Visual comparative-statics curves track the analytical reference.

**Profile.** `MODE = "SMOKE"` for quick iteration (3×128, ER 4000 / SHAC
800 steps). `MODE = "FULL"` for production-quality verification
(4×256, ER 8000 / SHAC 1600). Best step is restored from
`checkpoint_history` via the held-out MAE metric.

**This notebook is a self-contained training demo and is NOT a
dependency for NB09.** The companion Bayesian-inference notebook
`09_neural_surrogate_bayesian.ipynb` owns its own NN training
(panel-derived bounds → bespoke env → SHAC → MCMC), so there is no
weight handoff between the two. NB08 may be folded into NB09 (or
moved to an appendix) once the policy-based pipeline is validated.

(Originally a companion notebook `09_neural_surrogate_frictional.ipynb`
was planned for the full 5-D surrogate; that has been subsumed by NB09.)

---
# Section 0: Setup

In [ ]:
import sys, os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

_nb_file = globals().get("__vsc_ipynb_file__")
_REPO_ROOT = Path(_nb_file).parent.parent if _nb_file else Path.cwd().parent
sys.path.insert(0, str(_REPO_ROOT))

from src.v2.environments.basic_investment import (
    EconomicParams, ShockParams, compute_frictionless_policy,
)
from src.v2.environments.parameterized_basic_investment import (
    ParameterizedBasicInvestmentEnv,
)
from src.v2.estimation.beta_sampler import (
    BETA_DIM_NAMES, BetaSampler, DEFAULT_UNIFORM_BOUNDS,
)
from src.v2.evaluation.policies import restore_selected_snapshot
from src.v2.networks.policy import ParameterizedPolicyNetwork
from src.v2.networks.state_value import ParameterizedStateValueNetwork
from src.v2.trainers.config import (
    ERConfig, SHACConfig, NetworkConfig, OptimizerConfig,
)
from src.v2.trainers.er_param import train_er_param
from src.v2.trainers.shac_param import train_shac_param
from src.v2.utils.seeding import fold_in_seed

# -------------------------------------------------------------------
# Output directory for figures and tables.
# -------------------------------------------------------------------
FIG_DIR = _REPO_ROOT / "outputs" / "notebooks" / "NB08_neural_surrogate_validation"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("FIG_DIR =", FIG_DIR)

# -------------------------------------------------------------------
# Profile: SMOKE for quick iteration, FULL for production verification.
# -------------------------------------------------------------------
MODE = "FULL"     # set to "SMOKE" for fast smoke testing (~6 min)

PROFILE = {
    "SMOKE": dict(n_layers=3, n_neurons=128, er_steps=4000, shac_steps=800,
                  er_lr=3e-3,  shac_actor_lr=2e-3, shac_critic_lr=5e-3),
    "FULL":  dict(n_layers=4, n_neurons=256, er_steps=8000, shac_steps=1600,
                  er_lr=1e-3,  shac_actor_lr=2e-3, shac_critic_lr=5e-3),
}[MODE]
print("MODE =", MODE, " PROFILE =", PROFILE)
print("BETA dims:", BETA_DIM_NAMES)

In [ ]:
# ----- Calibrated (β-independent) params --------------------------------
nominal_econ   = EconomicParams(
    interest_rate=0.04, depreciation_rate=0.10, production_elasticity=0.5)
nominal_shocks = ShockParams(rho=0.5, sigma=0.24, mu=0.0)

env = ParameterizedBasicInvestmentEnv(
    nominal_econ=nominal_econ,
    nominal_shocks=nominal_shocks,
    k_min_mult=0.1, k_max_mult=8.0, z_sd_mult=3.0,
)
print(f"k_star (nominal) = {env.k_star:.3f},  k_min = {env.k_min:.3f},  k_max = {env.k_max:.3f}")
print(f"z_min = {env.z_min:.3f},  z_max = {env.z_max:.3f}")

# Training sampler — uniform over each β coordinate, friction dims frozen.
# Default uniform_bounds (see DEFAULT_UNIFORM_BOUNDS in beta_sampler.py)
# cover the validation-sweep ranges with margin.
beta_sampler = BetaSampler(freeze_dims=(3, 4))
anchor_beta  = beta_sampler.prior_mean()      # midpoint of uniform box
print("training uniform bounds =", beta_sampler.uniform_bounds)
print("anchor β (uniform-box midpoint, φ=0 slice) =", anchor_beta.numpy().ravel())

In [ ]:
# ----- Dataset (initial states for training) ----------------------------
N_TRAIN = 4096
ds_seed = tf.constant([20, 26], dtype=tf.int32)
train_dataset = {
    "s_endo": env.sample_initial_endogenous(N_TRAIN, seed=ds_seed),
    "z":      env.sample_initial_exogenous(
        N_TRAIN, seed=tf.constant([21, 26], dtype=tf.int32)),
}
print({k: v.shape for k, v in train_dataset.items()})

---
# Section 1: Train ER surrogate on the frictionless slice

Single-stage training over `(α, ρ, σ_ε)` only. No curriculum, no warm-start.

In [ ]:
# Held-out validation set (fixed once, used by eval_callback below).
N_VAL = 1024
val_s_endo = env.sample_initial_endogenous(
    N_VAL, seed=tf.constant([99, 99], dtype=tf.int32))
val_z      = env.sample_initial_exogenous(
    N_VAL, seed=tf.constant([100, 99], dtype=tf.int32))
val_beta   = beta_sampler.sample(
    N_VAL, seed=fold_in_seed((99, 99), "val", "beta"))
val_dataset = {"s_endo": val_s_endo, "z": val_z, "beta": val_beta}

def make_holdout_mae_callback():
    """eval_callback returning held-out MAE vs the analytical k'.

    Future-proofing: for the frictional notebook 09 (no closed form),
    swap the body to compute Euler residual on the same (s, β) tuples.
    Trainer / checkpoint plumbing is unchanged — early-stopping just
    monitors whichever key name the callback returns.
    """
    s_all   = env.merge_state(val_dataset["s_endo"], val_dataset["z"])
    k_all   = val_dataset["s_endo"][:, 0].numpy()
    kp_true = env.analytical_kprime(
        val_dataset["z"], val_dataset["beta"]).numpy().ravel()
    def cb(step, env, policy, value_net, val_dataset):
        a = policy(s_all, val_dataset["beta"]).numpy().ravel()
        kp_pred = np.clip(
            (1.0 - env.delta_rate) * k_all + a, env.k_min, env.k_max)
        return {"mae_holdout": float(np.mean(np.abs(kp_pred - kp_true)))}
    return cb

policy_er = ParameterizedPolicyNetwork(
    state_dim=env.state_dim(), action_dim=env.action_dim(), beta_dim=5,
    **env.action_spec(),
    n_layers=PROFILE["n_layers"], n_neurons=PROFILE["n_neurons"],
    seed=(101, 0))
policy_er(tf.zeros((1, env.state_dim())), tf.zeros((1, 5)))

er_ckpt = []   # filled by trainer via capture_checkpoint
er_config = ERConfig(
    n_steps=PROFILE["er_steps"], batch_size=256,
    eval_interval=200,
    master_seed=(20, 26),
    loss_type="crossprod",
    polyak_rate=0.995,
    network=NetworkConfig(
        n_layers=PROFILE["n_layers"], n_neurons=PROFILE["n_neurons"]),
    policy_optimizer=OptimizerConfig(learning_rate=PROFILE["er_lr"]),
    checkpoint_history=er_ckpt,
    snapshot_targets=("policy",),
    # Early stopping (plateau rule only — future-proof for nb 09 where
    # there is no analytical reference to threshold against).
    monitor="mae_holdout",
    mode="min",
    plateau_patience=5,
    plateau_rel_delta=0.02,
    min_steps_before_stop=1000,
)
result_er = train_er_param(
    env, policy_er, beta_sampler, train_dataset,
    val_dataset=val_dataset, config=er_config,
    eval_callback=make_holdout_mae_callback())
total_er_sec = result_er['wall_time_sec']
print(f"\nER wall time: {total_er_sec:.1f}s | {len(er_ckpt)} checkpoints captured")
print(f"ER stop reason: {result_er.get('stop_reason', 'max_steps')}")

---
# Section 2: Train SHAC surrogate on the frictionless slice

Single-stage training. Same `beta_sampler` (frictions frozen at 0).

In [ ]:
policy_shac = ParameterizedPolicyNetwork(
    state_dim=env.state_dim(), action_dim=env.action_dim(), beta_dim=5,
    **env.action_spec(),
    n_layers=PROFILE["n_layers"], n_neurons=PROFILE["n_neurons"],
    seed=(202, 0))
policy_shac(tf.zeros((1, env.state_dim())), tf.zeros((1, 5)))

value_shac = ParameterizedStateValueNetwork(
    state_dim=env.state_dim(), beta_dim=5,
    n_layers=PROFILE["n_layers"], n_neurons=PROFILE["n_neurons"],
    seed=(203, 0))
value_shac(tf.zeros((1, env.state_dim())), tf.zeros((1, 5)))

shac_ckpt = []
shac_config = SHACConfig(
    n_steps=PROFILE["shac_steps"], batch_size=64,
    horizon=64, short_horizon=16, n_critic=8,
    eval_interval=50,
    master_seed=(20, 26),
    normalize_rewards=False,
    reward_scale_override=0.05,
    network=NetworkConfig(
        n_layers=PROFILE["n_layers"], n_neurons=PROFILE["n_neurons"]),
    policy_optimizer=OptimizerConfig(learning_rate=PROFILE["shac_actor_lr"]),
    critic_optimizer=OptimizerConfig(learning_rate=PROFILE["shac_critic_lr"]),
    checkpoint_history=shac_ckpt,
    snapshot_targets=("policy",),
    monitor="mae_holdout",
    mode="min",
    plateau_patience=8,
    plateau_rel_delta=0.02,
    min_steps_before_stop=300,
)
result_shac = train_shac_param(
    env, policy_shac, value_shac, beta_sampler, train_dataset,
    val_dataset=val_dataset, config=shac_config,
    eval_callback=make_holdout_mae_callback())
total_shac_sec = result_shac['wall_time_sec']
print(f"\nSHAC wall time: {total_shac_sec:.1f}s | {len(shac_ckpt)} checkpoints captured")
print(f"SHAC stop reason: {result_shac.get('stop_reason', 'max_steps')}")

---
# Section 2.5: Restore best step + training-curve diagnostic

Each trainer captures policy weights at every eval interval into
`checkpoint_history`. The training-curve plot below shows the held-out
MAE per step; we then restore the weights from the step that minimizes
this metric — independent of the (noisy) final-step weights.

This mirrors `docs/01_basic_investment_benchmark.ipynb`'s pattern: a
validation curve over training steps + best-checkpoint selection. The
callback returns `mae_holdout`; for the future frictional notebook 09
the callback body swaps to Euler-residual on the same plumbing.

In [ ]:
def restore_best_step(label, policy, ckpt_history, history):
    """Pick step minimizing `mae_holdout` and restore those weights."""
    if not ckpt_history or "mae_holdout" not in history:
        print(f"[{label}] no checkpoints / metric history — keeping final weights")
        return None, None
    steps  = history["step"]
    vals   = history["mae_holdout"]
    best_i = int(np.argmin(vals))
    best_step = int(steps[best_i])
    final_mae = float(vals[-1]); best_mae = float(vals[best_i])
    restore_selected_snapshot(policy, ckpt_history, best_step)
    print(f"[{label}] best step = {best_step:4d}  (mae_holdout = {best_mae:.4f})")
    print(f"[{label}] final step = {int(steps[-1]):4d} (mae_holdout = {final_mae:.4f}) "
          f"— gap = {100*(final_mae-best_mae)/max(best_mae,1e-8):.1f}%")
    return best_step, best_mae

best_er_step,   _ = restore_best_step("ER",   policy_er,   er_ckpt,   result_er['history'])
best_shac_step, _ = restore_best_step("SHAC", policy_shac, shac_ckpt, result_shac['history'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, label, ckpt_step, history, color in zip(
    axes, ["ER", "SHAC"],
    [best_er_step, best_shac_step],
    [result_er['history'], result_shac['history']],
    ['C0', 'C1'],
):
    if "mae_holdout" not in history:
        ax.axis('off'); continue
    ax.plot(history['step'], history['mae_holdout'], 'o-', ms=3, color=color)
    if ckpt_step is not None:
        ax.axvline(ckpt_step, color='red', alpha=0.5, ls='--',
                   label=f"best step = {ckpt_step}")
        ax.legend(loc='best')
    ax.set_xlabel('step'); ax.set_ylabel('held-out MAE')
    ax.set_title(f"{label}: held-out MAE vs training step")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / f"training_curve_{MODE.lower()}.png", dpi=150, bbox_inches='tight')
plt.show()

---
# Section 3: Comparative-statics slices (φ = 0)

All slices hold (k, z) at (k\*(nominal), E[z]) and remaining β components at the
prior-mean anchor. Each curve sweeps one parameter; the analytical Strebulaev–Whited
closed form is overlaid as the reference.


In [ ]:
def _kprime_from_policy(policy, s, beta):
    """Action I → next capital k' = (1-δ)k + I (clipped to bounds)."""
    a = policy(s, beta).numpy().ravel()
    k = s.numpy()[:, 0]
    return np.clip((1.0 - env.delta_rate) * k + a, env.k_min, env.k_max)

def _kprime_analytical(s, beta):
    return env.analytical_kprime(s[:, 1:2], beta).numpy().ravel()

def _sweep(param_idx, grid, fixed_state, anchor_beta_1d, n_grid=80):
    """Sweep one parameter index over `grid`; broadcast s and anchor β to match."""
    beta = np.tile(anchor_beta_1d, (n_grid, 1)).astype(np.float32)
    beta[:, param_idx] = grid
    beta_tf = tf.constant(beta)
    s = tf.constant(np.tile(fixed_state, (n_grid, 1)).astype(np.float32))
    return s, beta_tf

fixed_state = np.array([env.k_star, float(np.exp(env.mu))])    # (k*, E[z])
anchor_1d   = anchor_beta.numpy().ravel()                       # (5,)
print("fixed state =", fixed_state)
print("anchor β    =", anchor_1d)

In [ ]:
n_grid = 100

# Per-slice anchors. For ρ we deliberately use z = 1.5·exp(μ) (above
# the singular point z = exp(μ) where ρ has no analytical effect at
# all). Other slices stay at the conventional (k*, E[z]) anchor.
anchor_rho_state = np.array([env.k_star, 1.5 * float(np.exp(env.mu))],
                            dtype=np.float32)

# Sweep grids span the FULL training range of each coordinate.
alpha_lo, alpha_hi = DEFAULT_UNIFORM_BOUNDS["alpha"]
rho_lo,   rho_hi   = DEFAULT_UNIFORM_BOUNDS["rho"]
sig_lo,   sig_hi   = DEFAULT_UNIFORM_BOUNDS["sigma_epsilon"]

sweeps = {
    # name:          (β-index, sweep grid,                          fixed (k, z) state)
    "alpha":         (0, np.linspace(alpha_lo, alpha_hi, n_grid), fixed_state),
    "rho":           (1, np.linspace(rho_lo,   rho_hi,   n_grid), anchor_rho_state),
    "sigma_epsilon": (2, np.linspace(sig_lo,   sig_hi,   n_grid), fixed_state),
}

results = {}
for name, (idx, grid, fixed_s) in sweeps.items():
    s_in, beta_in = _sweep(idx, grid.astype(np.float32), fixed_s, anchor_1d, n_grid=n_grid)
    kp_er    = _kprime_from_policy(policy_er,   s_in, beta_in)
    kp_shac  = _kprime_from_policy(policy_shac, s_in, beta_in)
    kp_true  = _kprime_analytical(s_in, beta_in)
    results[name] = {
        "grid": grid, "er": kp_er, "shac": kp_shac, "true": kp_true,
        "mae_er":   float(np.mean(np.abs(kp_er   - kp_true))),
        "mae_shac": float(np.mean(np.abs(kp_shac - kp_true))),
        "anchor":   fixed_s.tolist(),
        "xrange":   (grid.min(), grid.max()),
    }
for name, r in results.items():
    print(f"{name:15s}  anchor=(k={r['anchor'][0]:.2f}, z={r['anchor'][1]:.2f})  "
          f"MAE_ER={r['mae_er']:.4f}  MAE_SHAC={r['mae_shac']:.4f}")

In [ ]:
# State slices at the anchor β.
# z spans the FULL env support (uniform sampling range).
# k also spans the full env support so we see how the surrogate behaves
# across the training range. Analytical k'(z;β) is independent of k in
# the frictionless model — k-slice should be flat.
z_grid = np.linspace(env.z_min, env.z_max, n_grid).astype(np.float32)
s_z = tf.constant(np.stack(
    [np.full(n_grid, env.k_star, dtype=np.float32), z_grid], axis=-1))
beta_z = tf.constant(np.tile(anchor_1d, (n_grid, 1)).astype(np.float32))
kp_er_z    = _kprime_from_policy(policy_er,   s_z, beta_z)
kp_shac_z  = _kprime_from_policy(policy_shac, s_z, beta_z)
kp_true_z  = _kprime_analytical(s_z, beta_z)
results["z (anchor β)"] = {
    "grid": z_grid, "er": kp_er_z, "shac": kp_shac_z, "true": kp_true_z,
    "mae_er":   float(np.mean(np.abs(kp_er_z   - kp_true_z))),
    "mae_shac": float(np.mean(np.abs(kp_shac_z - kp_true_z))),
    "xrange":   (env.z_min, env.z_max),
}

k_grid = np.linspace(env.k_min, env.k_max, n_grid).astype(np.float32)
z_anchor = float(np.exp(env.mu))
s_k = tf.constant(np.stack(
    [k_grid, np.full(n_grid, z_anchor, dtype=np.float32)], axis=-1))
beta_k = tf.constant(np.tile(anchor_1d, (n_grid, 1)).astype(np.float32))
kp_er_k    = _kprime_from_policy(policy_er,   s_k, beta_k)
kp_shac_k  = _kprime_from_policy(policy_shac, s_k, beta_k)
kp_true_k  = _kprime_analytical(s_k, beta_k)
results["k (anchor β)"] = {
    "grid": k_grid, "er": kp_er_k, "shac": kp_shac_k, "true": kp_true_k,
    "mae_er":   float(np.mean(np.abs(kp_er_k   - kp_true_k))),
    "mae_shac": float(np.mean(np.abs(kp_shac_k - kp_true_k))),
    "xrange":   (env.k_min, env.k_max),
}
for name in ["z (anchor β)", "k (anchor β)"]:
    r = results[name]
    print(f"{name:18s}  MAE_ER={r['mae_er']:.4f}  MAE_SHAC={r['mae_shac']:.4f}")

In [ ]:
# Two-row figure with FIXED y-axis range [0, k_max] across all panels
# and per-panel x-axis matching the training-range / state-support of
# the swept variable. Variations look proportionally accurate; no
# auto-zoom artifact.
slice_order = [
    "alpha", "rho", "sigma_epsilon",
    "z (anchor β)", "k (anchor β)",
]
xlabels = {
    "alpha": r"$\alpha$", "rho": r"$\rho$",
    "sigma_epsilon": r"$\sigma_\varepsilon$",
    "z (anchor β)": r"$z$",
    "k (anchor β)": r"$k$",
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for ax, name in zip(axes, slice_order):
    r = results[name]
    ax.plot(r["grid"], r["true"], 'k--', lw=2, label="analytical (φ=0)")
    ax.plot(r["grid"], r["er"],   'C0',  lw=1.5,
            label=f"ER  (MAE={r['mae_er']:.3f})")
    ax.plot(r["grid"], r["shac"], 'C1',  lw=1.5,
            label=f"SHAC (MAE={r['mae_shac']:.3f})")
    # Fixed y-range across panels.
    ax.set_ylim(0, env.k_max)
    # Full x-range as recorded in results['xrange'].
    ax.set_xlim(*r['xrange'])
    ax.set_xlabel(xlabels[name]); ax.set_ylabel(r"$k'$")
    title = f"$k'$ vs {xlabels[name]}"
    if name == "rho":
        title += r"  (anchor $z = 1.5$ to avoid $\rho$-singular point)"
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
axes[5].axis("off")
plt.tight_layout()
fig.savefig(FIG_DIR / f"slices_{MODE.lower()}.png", dpi=150, bbox_inches='tight')
plt.show()

---
# Section 4: Summary

In [ ]:
# Summary table; also persisted as CSV alongside the figures.
import csv
k_range      = float(env.k_max - env.k_min)
action_range = float(env.I_max - env.I_min)
print(f"Action range = {action_range:.3f}, k range = {k_range:.3f}\n")
print(f"{'slice':18s} {'MAE_ER':>10s} {'MAE_SHAC':>10s}  {'ER % of k range':>16s}  {'SHAC % of k range':>18s}")
print("-" * 80)
rows = []
for name in ["alpha", "rho", "sigma_epsilon", "z (anchor β)", "k (anchor β)"]:
    r = results[name]
    er_pct   = 100.0 * r['mae_er']   / k_range
    shac_pct = 100.0 * r['mae_shac'] / k_range
    rows.append({
        "slice": name,
        "MAE_ER": r['mae_er'],
        "MAE_SHAC": r['mae_shac'],
        "ER_pct_k_range": er_pct,
        "SHAC_pct_k_range": shac_pct,
    })
    print(f"{name:18s} {r['mae_er']:10.4f} {r['mae_shac']:10.4f}  "
          f"{er_pct:15.2f}%  {shac_pct:17.2f}%")

print(f"\nTotal wall time — ER: {total_er_sec:.1f}s, SHAC: {total_shac_sec:.1f}s")

csv_path = FIG_DIR / f"slice_mae_{MODE.lower()}.csv"
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print(f"Wrote slice MAE table: {csv_path}")